In [ ]:
from adaptive_latents.input_sources.lds_simulation import LDS
from adaptive_latents.input_sources.kalman_filter import StreamingKalmanFilter
from adaptive_latents import StimRegressor, ArrayWithTime
from adaptive_latents.regressions import BaseMultiKernelRegressor
import numpy as np
from adaptive_latents import ArrayWithTime
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


rng = np.random.default_rng()

In [ ]:
def finalize_log(sr:StimRegressor, stim_intended_samples):
    error = ArrayWithTime.from_list(sr.log['pred_error'], drop_early_nans=True, squeeze_type='to_2d')
    sr.log['pred_error'] = error

    # pred_error_with_origin_t = [ArrayWithTime(p,t) for p, t in zip(sr.log['pred_error'], sr.log['pred_origin_t'])]
    # error_ot = ArrayWithTime.from_list(pred_error_with_origin_t, drop_early_nans=True, squeeze_type='to_2d')
    # sr.log['pred_error_ot'] = error_ot

    sr.log['stim_intended_samples'] = stim_intended_samples.slice((stim_intended_samples > 0).any(axis=1))
    return sr



In [ ]:
n_rotations = 100
noise_variance = 0.05
stims_per_rotation = 2
stim_magnitude = 10

In [ ]:

aa = np.linspace(-2, 5, 7)[1:2]
bb = np.linspace(-2, 5, 8)[1:2]
# aa = [-0.8333]
# bb = [-1]
cc = [None]
n_repeats = 2_000



mse_s = []
traces = []
rng = np.random.default_rng(5)

with tqdm(total=len(aa)*len(bb) * len(cc) * n_repeats) as pbar:
    for _ in range(n_repeats):
        _, Y, stim = LDS.run_nest_dynamical_system(n_rotations, stims_per_rotation=stims_per_rotation, stim_magnitude=stim_magnitude, rng=rng, u_function='curvy flips', noise=noise_variance)
        mse_s.append([])
        traces.append([])
        for a in aa:
            mse_s[-1].append([])
            traces[-1].append([])
            for b in bb:
                mse_s[-1][-1].append([])
                traces[-1][-1].append([])

                for c in cc:
                    sr = StimRegressor(
                        autoreg=StreamingKalmanFilter(),
                        stim_reg=BaseMultiKernelRegressor(length_scales=[10**a,1,10**b],kernel_weight_ratios=[1,1,1], maxlen=500),
                        log_level=2,
                        check_dt=True
                    )

                    sr.offline_run_on([(Y, 'X'), (stim, 'stim')], convinient_return=False, show_tqdm=False)
                    finalize_log(sr, stim)

                    flip_time = stim.t[stim.shape[0]//2]
                    pred_error = sr.log['pred_error'][:,2]
                    stim_pred_error = sr.log['pred_error'].slice_by_time(stim.t[np.squeeze(stim) == 1])[:,2]

                    pre_flip_error = np.nanmean(stim_pred_error.slice_by_time(slice(None, flip_time))**2)
                    post_flip_error = np.nanmean(stim_pred_error.slice_by_time(slice(flip_time, None))**2)

                    traces[-1][-1][-1].append(pred_error)
                    mse_s[-1][-1][-1].append((pre_flip_error, post_flip_error))

                    pbar.update(1)


all_mse_s = np.array(mse_s)

In [ ]:
10**aa, 10**bb


In [ ]:
s = (slice(None), slice(None), slice(None), slice(None), 1)
idx = np.unravel_index(np.argmin(all_mse_s[s]), all_mse_s[s].shape)
print(idx)
print(list(map(lambda x: x[0][x[1]], zip([aa,bb,cc],idx[1:]))))

fig, axs = plt.subplots(nrows=2, figsize=(10,5))
pred_error = traces[idx[0]][idx[1]][idx[2]][idx[3]]
axs[0].plot(pred_error)

axs[1].plot(pred_error.slice_by_time(slice(flip_time, None)))
axs[1].plot(pred_error.slice_by_time(slice(None,flip_time)))



fig, ax = plt.subplots()
for a in np.array(traces, dtype=object)[:,idx[1],idx[2],idx[3]]:
    ax.plot(a.t, a)



In [ ]:
%matplotlib inline

from scipy.signal import savgol_filter

trs = np.squeeze(np.array(traces,dtype=object))
lens = [len(x) for x in trs]
t = trs[np.argmax(lens)].t
maxlen = max(lens)
trs = [np.hstack([np.zeros(maxlen - len(x)) * np.nan, x]) for x in trs]
trs = np.array(trs)

fig, ax = plt.subplots()
mean_mse = np.nanmean(trs**2, axis=0)
smoothed_mean_mse = ArrayWithTime(savgol_filter(mean_mse, 40, 1),t)
ax.plot(t, mean_mse, label='raw')
ax.plot(t, smoothed_mean_mse, label='smoothed')
ax.set_xlabel('time (number of rotations)')
ax.set_ylabel('averaged MSE')

threshold = float(smoothed_mean_mse.slice_by_time(slice(15,45)).max())
ax.axhline(threshold, color='k', linestyle='--')


initial = smoothed_mean_mse.slice_by_time(slice(0,30))
initial_t = initial.t[np.nonzero(initial < threshold)[0][0]]
ax.axvline(initial_t, color='k', alpha=.5)
print(f'initial training in {initial_t :.1f} samples')

recovery = smoothed_mean_mse.slice_by_time(slice(50,80))
recovered_t = recovery.t[np.nonzero(recovery < threshold)[0][0]]
ax.axvline(recovered_t, color='k', alpha=.5)
print(f'recovered in {(recovered_t-50) :.1f} samples')

# ax.set_xlim([50, 55])



In [ ]:
smoothed_mean_mse.shape, mean_mse.shape

In [ ]:
all_mse_s.shape

In [ ]:
mse_s = all_mse_s[:,:,:,0,:].mean(axis=0)

vmin = np.log(mse_s.min())
vmax= np.log(mse_s.max())

fig, axs = plt.subplots(ncols=2, figsize=(12,5), layout='constrained')
axs[0].pcolormesh(bb, aa, np.log(mse_s[...,0]), shading='nearest', vmin=vmin, vmax=vmax)
axs[0].set_ylabel('log(length_scale)')
axs[0].set_xlabel('log(time_scale)')
axs[0].set_title('first half (pre switch)')
axs[0].set_xticks(bb)
axs[0].set_yticks(aa)

# axs[1].pcolormesh(bb, aa, np.log(mse_s.mean(axis=-1)), shading='nearest', vmin=vmin, vmax=vmax)
# axs[1].set_xlabel('b')
# axs[1].set_title('average')

cm = axs[1].pcolormesh(bb, aa, np.log(mse_s[...,1]), shading='nearest', vmin=vmin, vmax=vmax)
axs[1].set_ylabel('log(length_scale)')
axs[1].set_xlabel('log(time_scale)')
axs[1].set_title('second half error (post switch)')
axs[1].set_xticks(bb)
axs[1].set_yticks(aa)

fig.colorbar(mappable=cm)



In [ ]:
to_min = np.squeeze(all_mse_s.mean(axis=0)[...,1])
plt.matshow(to_min)

idx = np.unravel_index(np.argmin(to_min), to_min.shape)


In [ ]:
fig, axs = plt.subplots(ncols=3, figsize=(15,5))

to_dice = all_mse_s[...,0].mean(axis=0)

for i, values in enumerate([aa,bb,cc]):
    bests = np.min(to_dice, axis=tuple(set(range(3)) - {i}))
    axs[i].plot(values,bests)
    print(values[np.argmin(bests)])
    # axs[1].plot(bb,np.min(to_dice, axis=(0,2)))
    # axs[2].plot(cc,np.min(to_dice, axis=(0,1)))



In [ ]:
# %load_ext autoreload
# %autoreload 2

rng = np.random.default_rng()
_, Y, stim = LDS.run_nest_dynamical_system(n_rotations, stims_per_rotation=stims_per_rotation, stim_magnitude=stim_magnitude, rng=rng, u_function='curvy', noise=noise_variance)


a = .9
sr = StimRegressor(
    autoreg=StreamingKalmanFilter(),
    stim_reg=BaseMultiKernelRegressor(length_scales=[10**-.43,1,10**15], maxlen=500),
    log_level=2,
    check_dt=True
)

sr.offline_run_on([(Y, 'X'), (stim, 'stim')], convinient_return=False, show_tqdm=False)
finalize_log(sr, stim)

flip_time = stim.t[stim.shape[0]//2]
pred_error = sr.log['pred_error'][:,2]
stim_pred_error = sr.log['pred_error'].slice_by_time(stim.t[np.squeeze(stim) == 1])[:,2]

pre_flip_error = np.nanmean(stim_pred_error.slice_by_time(slice(None, flip_time))**2)
post_flip_error = np.nanmean(stim_pred_error.slice_by_time(slice(flip_time, None))**2)

fig, ax = plt.subplots()
ax.plot(pred_error)
ax.set_ylim([-5,5]);

In [ ]:
from adaptive_latents import datasets
print(datasets.Odoherty21Dataset().neural_data.shape)